In [ ]:
!pip install torch_geometric6

In [ ]:
cd ..

In [ ]:
# from torch_geometric.loader import DataLoader
from torch.utils.data import DataLoader
from src.sparse_dataloader import TrafficDataset

In [ ]:
%%time

processed_data_dir = "data/datos_procesados"
sparse_data_dir = f"{processed_data_dir}/sparse_datasets"
graph_filename = f"{processed_data_dir}/conectivity_graph/road_graph.pickle"
static_data_filename = f"{sparse_data_dir}/static_dataset.h5"
dynamic_data_filenames = [
    f"{sparse_data_dir}/coi_points_dataset.h5",
    f"{sparse_data_dir}/coi_tracts_dataset.h5",
    f"{sparse_data_dir}/irregularities_dataset.h5",
    f"{sparse_data_dir}/events_dataset.h5",
    f"{sparse_data_dir}/incidents_dataset.h5",
]
metadata_filenames = [
    f"{sparse_data_dir}/coi_points_dataset_metadata.pkl",
    f"{sparse_data_dir}/coi_tracts_dataset_metadata.pkl",
    f"{sparse_data_dir}/irregularities_dataset_metadata.pkl",
    f"{sparse_data_dir}/events_dataset_metadata.pkl",
    f"{sparse_data_dir}/incidents_dataset_metadata.pkl",
]
target_data_filename = f"{sparse_data_dir}/jams_dataset.h5"
in_timesteps = 150
out_timesteps = 50

traffic_dataset = TrafficDataset(
    graph_path=graph_filename,
    static_data_path=static_data_filename,
    dynamic_data_paths=dynamic_data_filenames,
    metadata_paths=metadata_filenames, 
    target_data_path=target_data_filename,
    in_timesteps=in_timesteps,
    out_timesteps=out_timesteps,
    future_features=[
        "coi_puntos",
        "coi_tramo",
        "AGLOMERACION",
        "EMERGENCIA",
        "MANIFESTACION",
    ],
    node_batch_size=1024,
)

In [ ]:
result = traffic_dataset._get_time_and_node_indices(0)

In [ ]:
import torch

def pad_collate(batch):
    assert len(batch) > 0, "Empty batch."

    # --- Sanity: all samples must share same node set and edges ---
    N0 = batch[0].x_static.size(0)
    E0 = batch[0].edge_index.size(1)
    edge0 = batch[0].edge_index

    for i, d in enumerate(batch[1:], start=1):
        if d.x_static.size(0) != N0:
            raise ValueError(f"Item {i} has N={d.x_static.size(0)} but first has N={N0}.")
        if d.edge_index.size(1) != E0:
            raise ValueError(f"Item {i} has E={d.edge_index.size(1)} but first has E={E0}.")
        # Require identical edges (same ordering). If you only need same *set* of edges,
        # customize this check.
        if not torch.equal(d.edge_index, edge0):
            raise ValueError(f"Item {i} edge_index differs from first item.")

    # --- Always present ---
    x_static = torch.stack([d.x_static for d in batch], dim=0)  # (B, N, F_static)
    y        = torch.stack([d.y        for d in batch], dim=0)  # (B, N, Tout)

    # --- Dynamic features: support either split past/future or a single x_dynamic ---
    out = {
        "x_static":   x_static,
        "y":          y,
        "edge_index": edge0,     # (2, E) shared across batch
        "num_nodes":  N0,
        "num_edges":  E0,
    }

    has_past   = hasattr(batch[0], "x_dynamic_past")
    has_future = hasattr(batch[0], "x_dynamic_future")
    has_single = hasattr(batch[0], "x_dynamic")

    if has_past and has_future:
        out["x_dynamic_past"]   = torch.stack([d.x_dynamic_past   for d in batch], dim=0)  # (B, N, Tin, F_dyn)
        out["x_dynamic_future"] = torch.stack([d.x_dynamic_future for d in batch], dim=0)  # (B, N, Tout, F_future)
    elif has_single:
        out["x_dynamic"] = torch.stack([d.x_dynamic for d in batch], dim=0)  # (B, N, Tin, F_dyn)
    else:
        # If your dataset always has dynamics, you can raise instead:
        # raise AttributeError("Expected x_dynamic_past/x_dynamic_future or x_dynamic in Data.")
        pass

    # Optional: if your Data carries a node id tensor you want to pass through
    if hasattr(batch[0], "node_indices"):
        out["node_indices"] = batch[0].node_indices  # (N,) shared across items

    return out

In [ ]:
batch_size = 2

traffic_dataloader = DataLoader(
    traffic_dataset,           # created with node_batch_size=... (your code supports this) :contentReference[oaicite:3]{index=3}
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    collate_fn=pad_collate,
)

In [ ]:
for batch in traffic_dataloader:
    a = 1
    break

In [ ]:
batch.keys()

In [ ]:
batch['x_static'].shape,batch['y'].shape,batch['edge_index'].shape,batch['num_nodes'],batch['num_edges'],batch['x_dynamic_past'].shape,batch['x_dynamic_future'].shape

In [ ]:
DataBatch(edge_index=[2, 5096], y=[2,618, 3], x_static=[1236, 117], x_dynamic_past=[1236, 6, 112], x_dynamic_future=[1236, 3, 5], num_nodes=278214, batch=[278214], ptr=[3])

In [ ]:
len(list(traffic_dataset.get_roads_within_hops().keys()))

In [ ]:
278214/2